[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# AsyncConnection &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `ask` and `rounded`. Run it first. Each task opens the
connections it needs and closes them, so they can be run in any order.


In [1]:
import asyncio
import gc
import getpass
import os
import subprocess
import sys
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

async def ask(conn, label, seconds=0.3):
    """One slow query, so that waiting is visible."""
    cur = await conn.execute("SELECT pg_sleep(%s), %s", (seconds, label))
    return (await cur.fetchone())[1]


def rounded(seconds):
    """A tenth of a second, which is as precise as a timing here can honestly be."""
    return f"{seconds:.1f}s"


async def warned_about(work):
    """Run something that warns, and give back the warnings without the file paths they carry."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        await work()
        gc.collect()                                          # a lost coroutine warns when collected
    return [f"{w.category.__name__}: {w.message}" for w in caught]


print("server:", start_server())
print(report())
print("an event loop is running already:", asyncio.get_running_loop().is_running())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
an event loop is running already: True


**1.** Before and after the `await`.


In [2]:
conn = await psycopg.AsyncConnection.connect("dbname=guide")

pending = conn.execute("SELECT count(*) FROM events")
print("before awaiting:", type(pending).__name__)

cur = await pending
print("after awaiting: ", type(cur).__name__, await cur.fetchone())
await conn.close()


before awaiting: coroutine
after awaiting:  AsyncCursor (5000,)


The coroutine is a description of work. Nothing has reached the server until the `await`, which is
the whole reason a missing one is silent.


**2.** The same two queries, shared and not.


In [3]:
one = await psycopg.AsyncConnection.connect("dbname=guide")
two = await psycopg.AsyncConnection.connect("dbname=guide")


async def timed(*work):
    start = time.perf_counter()
    await asyncio.gather(*work)
    return rounded(time.perf_counter() - start)


print("both on one connection: ", await timed(ask(one, "a"), ask(one, "b")))
print("one connection each:    ", await timed(ask(one, "a"), ask(two, "b")))
await one.close()
await two.close()


both on one connection:  0.6s
one connection each:     0.3s


Two queries of three tenths of a second. Sharing adds them up, and a connection each overlaps them.
The `gather` is identical in both lines.


**3.** `async for` over a server-side cursor.


In [4]:
conn = await psycopg.AsyncConnection.connect("dbname=guide")

async with conn.cursor(name="ten") as cur:
    await cur.execute("SELECT id, kind FROM events ORDER BY id LIMIT 10")
    async for row in cur:
        print(row)

await conn.close()


(1, 'view')
(2, 'purchase')
(3, 'click')
(4, 'view')
(5, 'purchase')
(6, 'click')
(7, 'view')
(8, 'purchase')
(9, 'click')
(10, 'view')


A named cursor leaves the rows on the server and fetches them in batches, so the loop over it has to
be able to wait, which is what `async for` is.


**4.** A limit the server enforces.


In [5]:
conn = await psycopg.AsyncConnection.connect("dbname=guide")
await conn.execute("SET statement_timeout = '250ms'")

try:
    await conn.execute("SELECT pg_sleep(3)")
except errors.QueryCanceled as error:
    print("stopped:", error)

await conn.rollback()                                               # the transaction failed with it
print("usable again:", conn.info.transaction_status.name)
await conn.execute("SET statement_timeout = 0")
await conn.close()


stopped: canceling statement due to statement timeout
usable again: IDLE


The rollback is the part that is easy to leave out. A canceled statement is a failed statement, and
until the transaction is rolled back every later query on that connection fails too.


**5.** Canceling from Python.


In [6]:
conn = await psycopg.AsyncConnection.connect("dbname=guide")

start = time.perf_counter()
task = asyncio.create_task(conn.execute("SELECT pg_sleep(4)"))
await asyncio.sleep(0.3)
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print("it ran for", rounded(time.perf_counter() - start), "rather than 4 seconds")

await conn.rollback()
print("transaction:", conn.info.transaction_status.name)
await conn.close()


it ran for 0.3s rather than 4 seconds
transaction: IDLE


psycopg sends the server a cancellation, so this is not only Python walking away: the query stops on
the server too. Without that, a canceled task would leave the database working on an answer nobody
would ever read.


**6.** Three counts at once.


In [7]:
async def counted(sql):
    """A connection of its own, and it closes when the answer is in."""
    conn = await psycopg.AsyncConnection.connect("dbname=guide")
    try:
        cur = await conn.execute(sql)
        return (await cur.fetchone())[0]
    finally:
        await conn.close()


totals = await asyncio.gather(
    counted("SELECT count(*) FROM events"),
    counted("SELECT count(*) FROM events WHERE kind = 'click'"),
    counted("SELECT count(DISTINCT kind) FROM events"))
print("all:", totals[0], " clicks:", totals[1], " kinds:", totals[2])
print("one wait, not three")


all: 5000  clicks: 1666  kinds: 3
one wait, not three


The `finally` matters more here than in a synchronous version, because three connections opened in a
`gather` are three connections to lose if one of the queries raises. **Connection Pools** is the
notebook where handing them back stops being your job.


---

&#8592; **Back to:** [AsyncConnection](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/10-async-connection.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
